# set 6

## 1) 데이터 및 시나리오

### 교육 수강자 분석

> 삼성전자 임직원의 체계적인 커리어 패스 설계를 위해 데이터 분석과 같은 Hard Skill 역량과 Soft Skill 역량을 조화롭게 함양하기 위해 HRD 부서의 조프로는 전산팀의 협조를 받아 데이터를 확보하였다.

※ 분석을 수행하기 전, 상기 데이터를 이용하여 아래의 전처리를 수행하시오.

- **단계 1:** 분석에 사용하지 않을 `city`, `company_size`, `company_type` 컬럼을 제거하시오.
- **단계 2:** 각 문자형 변수에 결측치가 하나라도 존재하는 행은 모두 제거하시오.
- **단계 3:** `experience` 변수 값이 `'>20'` 또는 `'<1'`이 있는 행을 제거하고 `experience` 변수의 유형을 정수로 변환하시오.
- **단계 4:** `last_new_job` 변수의 값이 `'>4'` 또는 `'never'`인 행을 제거하고 해당 변수의 유형을 정수로 변환하시오.

전처리 수행 이후 행 개수는 **7,522**이며 전처리가 완료된 데이터를 `base` 객체로 지정하고 이를 사용하여 문제를 풀이하시오.

### 데이터 개요

| 파일명 | 행 | 열 | 인코딩 |
|---|---:|---:|---|
| `edu_enrollees.csv` | 19158 | 15 | UTF-8 |

---

## 1) 데이터 및 시나리오

### 변수 상세

| 변수명 | 유형 | 설명 |
|---|---|---|
| `enrollee_id` | int | 수료자 ID |
| `city` | string | 도시 코드 |
| `city_development_index` | float | 도시 발전 지표 |
| `gender` | string | 성별 |
| `relevant_experience` | string | 관련 분야 경험 여부 |
| `enrolled_university` | string | 수강 과목명 |
| `education_level` | string | 학력 |
| `major_discipline` | string | 전공 |
| `experience` | string | 경력 |
| `company_size` | string | 현 직장 직원 수 |

### 변수 상세

| 변수명 | 유형 | 설명 |
|---|---|---|
| `company_size` | string | 현 직장 직원 수 |
| `company_type` | string | 현 직장 유형 |
| `last_new_job` | string | 전 직장 근속연수 |
| `training_hours` | int | 수료 시간 |
| `target` | int | 전배 희망 여부 `(0: 비희망, 1: 희망)` |
| `Xgrp` | string | Train/Test Set 구분 |


## 2) 문제

### 필요 라이브러리 함수 및 클래스 목록

| 목록 |
|---|
| `from sklearn.linear_model import LogisticRegression` |
| `from sklearn.neighbors import KNeighborsClassifier` |


In [8]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier

df = pd.read_csv("../../dataset/edu_enrollees.csv")
#SCDI
display(df.shape)
display(df.columns)
display(df.dtypes)
display(df.isna().sum())

(19158, 15)

Index(['enrollee_id', 'city', 'city_development_index', 'gender',
       'relevant_experience', 'enrolled_university', 'education_level',
       'major_discipline', 'experience', 'company_size', 'company_type',
       'last_new_job', 'training_hours', 'target', 'Xgrp'],
      dtype='object')

enrollee_id               float64
city                       object
city_development_index    float64
gender                     object
relevant_experience        object
enrolled_university        object
education_level            object
major_discipline           object
experience                 object
company_size               object
company_type               object
last_new_job               object
training_hours            float64
target                    float64
Xgrp                       object
dtype: object

enrollee_id                  0
city                         0
city_development_index       0
gender                    4508
relevant_experience          0
enrolled_university        386
education_level            460
major_discipline          2813
experience                  65
company_size              5938
company_type              6140
last_new_job               423
training_hours               0
target                       0
Xgrp                         0
dtype: int64

In [22]:
#단계1
cols = ['city','company_size', 'company_type']
df_1 = df.drop(columns = cols).copy()
display(df_1.shape)
#단계2
df_2 = df_1.dropna().copy()
display(df_2.shape)
display(df_2['experience'].unique())
#단계3
cond1 = df_2['experience'].isin(['>20','<1'])
df_3 = df_2.loc[~cond1,:].copy()
df_3['experience'] = df_3['experience'].astype(int)
display(df_3['experience'].unique())
#단계4
cond2 = df_3['last_new_job'].isin(['>4','never'])
df_4 = df_3.loc[~cond2,:].copy()
df_4['last_new_job'] = df_4['last_new_job'].astype(int)
display(df_4['last_new_job'].unique())

base = df_4.copy()
display(base.shape)

(19158, 12)

(12477, 12)

array(['>20', '15', '13', '7', '5', '16', '4', '11', '<1', '18', '19',
       '12', '10', '9', '2', '6', '14', '3', '8', '20', '17', '1'],
      dtype=object)

array([15, 13,  7,  5, 16,  4, 11, 18, 19, 12, 10,  9,  2,  6, 14,  3,  8,
       20, 17,  1])

array([1, 2, 3, 4])

(7522, 12)


---

### Q01.

`base`를 사용하여 관련 분야 경험 여부(`relevant_experience`)에 따른 전배 희망 여부(`target`)를 기술통계량으로 확인하고자 한다.

관련 분야 경험이 없는 수료자 중 전배를 희망하는 수료자의 비율을 **A**,  
관련 분야 경험이 있는 수료자 중 전배를 희망하는 수료자의 비율을 **B**라 할 때,  
**A/B를 구하시오.**

※ 관련 경험이 없는 사람은 `relevant_experience` 변수의 값이 `'No relevant experience'`인 사람으로 정의한다.  
※ 관련 경험이 있는 사람은 `relevant_experience` 변수의 값이 `'Has relevant experience'`인 사람으로 정의한다.  
※ 정답은 반올림하여 소수점 둘째 자리까지 출력하시오. `(정답 예시: 0.12)`

In [43]:
df_q1 = base.copy()
def ratio(ser) :
    display(ser.value_counts())
    ser_1 = ser.value_counts()[1.0]
    display(ser_1,ser.shape[0])
    return ser_1/ser.shape[0]

df_q1_r = df_q1.groupby('relevant_experience')['target'].apply(ratio)
display(df_q1_r)
A = df_q1_r['No relevant experience']
B = df_q1_r['Has relevant experience']
display(round(A/B,2))

0.0    4790
1.0    1319
Name: Has relevant experience, dtype: int64

1319

6109

0.0    872
1.0    541
Name: No relevant experience, dtype: int64

541

1413

relevant_experience
Has relevant experience    0.215911
No relevant experience     0.382873
Name: target, dtype: float64

1.77


### Q02.

`base`를 사용하여 전배 희망 여부(`target`)에 영향을 주는 변수들을 확인하고자 한다.  
다음 절차에 따라 로지스틱 회귀분석을 수행하고 질문에 답하시오.

#### 단계 1

`gender`, `relevant_experience`, `enrolled_university`, `education_level`, `major_discipline` 변수로부터 더미 변수들을 생성한다.

단, 각 변수로부터 더미 변수를 생성할 때 마지막으로 등장하는 범주는 제외하도록 한다.

여기서 마지막으로 등장하는 범주란, 각 컬럼의 값을 사전 순으로 나열하였을 때 마지막으로 등장하는 값이다.

예를 들어, `col` 변수의 범주가 `['A', 'C', 'B', 'B', 'C', 'A']`의 값을 가진다면 사전 상의 마지막 값인 `C`가 제외된다.

#### 단계 2

단계 1에서 생성한 더미 변수와 `city_development_index`, `experience`, `last_new_job`, `training_hours`, `target`, `Xgrp` 변수를 결합하여 새로운 데이터셋을 구성한다.

- 데이터셋명: `job2`
- 이 데이터셋은 문제 3에서도 활용

이 때, `target`, `Xgrp`를 제외한 데이터셋의 컬럼은 아래 순서에 따르도록 한다.

1. `city_development_index`
2. `experience`
3. `last_new_job`
4. `training_hours`
5. `gender`의 더미 변수
6. `relevant_experience`의 더미 변수
7. `enrolled_university`의 더미 변수
8. `education_level`의 더미 변수
9. `major_discipline`의 더미 변수

#### 단계 3

단계 2에서 구성한 데이터셋 `job2`로 다음 조건에 따라 상수항(`Intercept`)이 포함된 로지스틱 회귀분석을 수행한다.

- 종속 변수: `target`
- 독립 변수(총 16개): `target`과 `Xgrp`를 제외한 나머지 변수
- 회귀식에 포함되는 독립 변수의 순서를 컬럼의 순서와 일치시킨다.

**상수항을 제외한 나머지 변수들에 대한 Odds Ratio 중 가장 큰 값을 기술하시오.**

$$
x_i \text{의 Odds Ratio}
=
\frac{
odds(P(Y=1|x_1,\cdots,x_i+1,\cdots,x_n))
}{
odds(P(Y=1|x_1,\cdots,x_i,\cdots,x_n))
}
$$

※ `LogisticRegression()` 클래스의 인자 `C`는 `100000`, `max_iter=1000`, `solver='liblinear'`으로 지정하시오.  
※ `LogisticRegression()` 클래스의 인자 `random_state`는 `123`으로 지정하시오.  
※ 정답은 소수점 셋째 자리에서 버림하여 둘째 자리까지 출력하시오. `(정답 예시: 0.12)`

---

In [62]:
"""
['enrollee_id', 'city', 'city_development_index', 'gender',
       'relevant_experience', 'enrolled_university', 'education_level',
       'major_discipline', 'experience', 'company_size', 'company_type',
       'last_new_job', 'training_hours', 'target', 'Xgrp']
"""
df_q2 = base.copy()
cols_q2 = ['gender','relevant_experience', 'enrolled_university', 'education_level','major_discipline']
df_q2_dummies = pd.get_dummies(df_q2,columns=cols_q2).copy()

list =[]
for col in cols_q2 :
    ser = pd.Series(df_q2[col].unique()).sort_values(ascending = False).reset_index(drop=True)
    display(col, ser)
    list.append(col+'_'+ser[0])
display(list)
display(df_q2_dummies.columns)
df_q2_1 = df_q2_dummies.drop(columns=list)
display(df_q2_dummies.shape, df_q2_1.shape)

#단계2
list2 = df_q2_1.columns
display(list2)
cols_job2 = ['city_development_index','experience','last_new_job', 'training_hours','gender_Female', 'gender_Male',
       'relevant_experience_Has relevant experience',
       'enrolled_university_Full time course',
       'enrolled_university_Part time course', 'education_level_Graduate',
       'education_level_Masters', 'major_discipline_Arts',
       'major_discipline_Business Degree', 'major_discipline_Humanities',
       'major_discipline_No Major', 'major_discipline_Other','target', 'Xgrp']
job2 = df_q2_1[cols_job2]

#단계3
display(job2.shape)

'gender'

0     Other
1      Male
2    Female
dtype: object

'relevant_experience'

0     No relevant experience
1    Has relevant experience
dtype: object

'enrolled_university'

...

Index(['enrollee_id', 'city_development_index', 'experience', 'last_new_job',
       'training_hours', 'target', 'Xgrp', 'gender_Female', 'gender_Male',
       'gender_Other', 'relevant_experience_Has relevant experience',
       'relevant_experience_No relevant experience',
       'enrolled_university_Full time course',
       'enrolled_university_Part time course',
       'enrolled_university_no_enrollment', 'education_level_Graduate',
       'education_level_Masters', 'education_level_Phd',
       'major_discipline_Arts', 'major_discipline_Business Degree',
       'major_discipline_Humanities', 'major_discipline_No Major',
       'major_discipline_Other', 'major_discipline_STEM'],
      dtype='object')

(7522, 24)

(7522, 19)

Index(['enrollee_id', 'city_development_index', 'experience', 'last_new_job',
       'training_hours', 'target', 'Xgrp', 'gender_Female', 'gender_Male',
       'relevant_experience_Has relevant experience',
       'enrolled_university_Full time course',
       'enrolled_university_Part time course', 'education_level_Graduate',
       'education_level_Masters', 'major_discipline_Arts',
       'major_discipline_Business Degree', 'major_discipline_Humanities',
       'major_discipline_No Major', 'major_discipline_Other'],
      dtype='object')

(7522, 18)

In [80]:

#D
cols_X = ['city_development_index','experience','last_new_job', 'training_hours','gender_Female', 'gender_Male',
       'relevant_experience_Has relevant experience',
       'enrolled_university_Full time course',
       'enrolled_university_Part time course', 'education_level_Graduate',
       'education_level_Masters', 'major_discipline_Arts',
       'major_discipline_Business Degree', 'major_discipline_Humanities',
       'major_discipline_No Major', 'major_discipline_Other']
cols_y = 'target' 
X = job2[cols_X].copy()
y = job2[cols_y].copy()
display(job2.dtypes)
#N
#M
model = LogisticRegression(C=100000,max_iter=1000,solver='liblinear',random_state=123)
model.fit(X,y)
display(model.coef_)
display(np.exp(model.coef_[0]))
ser_q2 = pd.Series(np.exp(model.coef_[0]), index = X.columns)
display(ser_q2.idxmax(),ser_q2.max())
display(np.floor(ser_q2.max()*100)/100)
#E

city_development_index                         float64
experience                                       int32
last_new_job                                     int32
training_hours                                 float64
gender_Female                                    uint8
gender_Male                                      uint8
relevant_experience_Has relevant experience      uint8
enrolled_university_Full time course             uint8
enrolled_university_Part time course             uint8
education_level_Graduate                         uint8
education_level_Masters                          uint8
major_discipline_Arts                            uint8
major_discipline_Business Degree                 uint8
major_discipline_Humanities                      uint8
major_discipline_No Major                        uint8
major_discipline_Other                           uint8
target                                         float64
Xgrp                                            object
dtype: obj

array([[-6.16042356e+00, -2.84523178e-02,  9.51256118e-02,
        -9.32104029e-04, -1.77221690e-01, -1.43611016e-01,
        -7.67253194e-01,  5.15739072e-01, -2.82857983e-01,
         2.58989370e-01, -4.40244893e-02,  2.90013957e-01,
         1.18976977e-01,  2.47072131e-01,  4.04528542e-01,
        -4.45273205e-01]])

array([0.00211136, 0.97194864, 1.09979699, 0.99906833, 0.83759408,
       0.86622463, 0.46428662, 1.6748759 , 0.75362681, 1.29562003,
       0.95693052, 1.33644614, 1.12634399, 1.28027146, 1.49859581,
       0.64064922])

'enrolled_university_Full time course'

1.6748758983036531

1.67

### Q03.

`job2`를 이용하여 전체 데이터를 Train과 Test Set으로 나누고, Train Set으로 학습한 모델을 Test Set에 적용하여 모델을 평가하고자 한다.

다음 절차에 따라 분석을 수행하고 질문에 답하시오.

#### 단계 1

2단계에서 구성한 데이터셋 `job2`에서 `Xgrp` 컬럼의 값이 `'train'`인 경우 Train Set으로, `'test'`인 경우 Test Set으로 정의하여 분할한다.

#### 단계 2

아래 가이드에 따라 Train Set으로 K-NN 분류 모델을 학습하고, 이 모델을 Test Set에 적용한다.

- 종속 변수: 전배 희망 여부(`target`)
- 독립 변수(총 16개): 전배 희망 여부(`target`)와 Train/Test set 구분 변수(`Xgrp`)를 제외한 모든 변수
- Euclidean 거리 기준 가장 가까운 5개 데이터의 전배 희망 여부(`target`)를 활용하여 예측

#### 단계 3

예측 결과를 바탕으로 아래 정의된 지표 **A**를 계산하여 기술하시오.

$$
A =
\frac{
(\# \text{ of true positive}) + (\# \text{ of true negative})
}{
(\# \text{ of total data})
}
$$

※ 정답은 반올림하여 소수점 둘째 자리까지 출력하시오. `(정답 예시: 0.12)`

In [102]:
#D
df_q3 = job2.copy()
train = df_q3.loc[df_q3['Xgrp'] == 'train', :].copy()
train_X = train.drop(columns=['target','Xgrp']).copy()
train_y = train['target'].copy()

test = df_q3.loc[df_q3['Xgrp'] == 'test', :].copy()
test_X = test.drop(columns=['target','Xgrp']).copy()
test_y = test['target'].copy()
#N
#M
model = KNeighborsClassifier(n_neighbors=5)
model.fit(train_X,train_y)
pred_y = model.predict(test_X)
#E

df_q3_2 = pd.DataFrame(test_y)
df_q3_2['pred'] = pred_y
display(df_q3_2)
display(round(df_q3_2.loc[df_q3_2['target'] == df_q3_2['pred'], :].shape[0]/df_q3_2.shape[0],2))

,target,pred
12001,0.0,0.0
12002,1.0,0.0
12003,0.0,0.0
12004,0.0,0.0
12008,0.0,0.0
...,...,...
19149,1.0,0.0
19150,0.0,0.0
19152,0.0,0.0
19153,1.0,0.0


0.71